# 03 · Train-only feature selection and PCA

Reads the tables and figures written by `python -m ssn select` and `python -m ssn pca`. All selectors and PCA objects were fitted on the training split, inside 5-fold stratified cross-validation where a comparison was made. The test split was never opened.

**Not a model claim.** The numbers here compare *selector settings* with a cheap reference estimator; candidate models are compared in Milestone 5.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from ssn.paths import repo_root

ROOT = repo_root()
T, F = ROOT / 'reports' / 'tables', ROOT / 'reports' / 'figures'
decision = json.loads((T / 'selection_decision.json').read_text())
decision

## 1. CV comparison over the k grid (filter vs embedded)

In [ ]:
cv = pd.read_csv(T / 'selection_cv_by_k.csv')
cv.round(4)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
for kind, g in cv.groupby('kind'):
    x = [decision['n_transformed_features'] if k == 'all' else int(k) for k in g['k']]
    ax.errorbar(x, g['pr_auc_mean'], yerr=g['pr_auc_std'], marker='o', capsize=3, label=f"{kind} ({g['method'].iloc[0]})")
ax.axhline(decision['best_pr_auc_mean'] - decision['cv_pr_auc_std'] * 0 , color='none')
ax.set_xscale('log'); ax.set_xlabel('k (transformed features kept, log scale)'); ax.set_ylabel('CV PR-AUC (mean ± std)')
ax.set_title('Selector setting vs CV PR-AUC, reference = class-weighted logistic regression (train only)')
ax.legend(); plt.show()

## 2. Descriptive rankings on the training split

Scores are fitted on the full training split for description only; `selected_frequency_top20` is how often the feature was among the top 20 across the 5 CV folds (stability).

In [ ]:
emb = pd.read_csv(T / 'selection_embedded_scores.csv')
emb.head(30)

In [ ]:
flt = pd.read_csv(T / 'selection_filter_scores.csv')
flt.head(30)

Source-column view: how many transformed features per original column appear in each top-30.

In [ ]:
pd.concat({'embedded_top30': emb.head(30)['source_column'].value_counts(), 'filter_top30': flt.head(30)['source_column'].value_counts()}, axis=1).fillna(0).astype(int)

## 3. PCA: explained variance and CV comparison

In [ ]:
display(Image(str(F / 'pca_scree.png')))
ev = pd.read_csv(T / 'pca_explained_variance.csv')
print('components to reach 95% variance:', int(ev.index[ev.cumulative >= 0.95][0] + 1), 'of', len(ev))
ev.head(10)

In [ ]:
pd.read_csv(T / 'pca_vs_nopca_cv.csv').round(4)

In [ ]:
display(Image(str(F / 'pca_2d_train.png')))

## 4. Reading the results

See section 11 of `reports/eda_feature_engineering_report.md` for the written interpretation, decision, and limitations.